# Celda 1: Importar librerías y cargar los datos
Primero, traemos las herramientas necesarias y el dataset de casas de California.

In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import joblib
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Importar los modelos que vamos a comparar
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Cargar el dataset de California Housing
print("Cargando datos...")
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target

# Dividir en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar los datos (Muy importante para modelos como SVM o Regresión Lineal)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Datos listos para entrenar. Dimensiones de X_train:", X_train_scaled.shape)

Cargando datos...
Datos listos para entrenar. Dimensiones de X_train: (16512, 8)


# Celda 2: Entrenamiento y seguimiento con MLflow
Aquí creamos un diccionario con los 5 modelos que pediste. Iteramos sobre ellos, los entrenamos y usamos MLflow para registrar los resultados de cada uno de forma automática.

In [0]:
# Definir los modelos a probar
modelos = {
    "Linear_Regression": LinearRegression(),
    "Decision_Tree": DecisionTreeRegressor(random_state=42),
    "Random_Forest": RandomForestRegressor(n_estimators=50, random_state=42), # 50 árboles para que corra rápido en clase
    "Gradient_Boosting": GradientBoostingRegressor(n_estimators=50, random_state=42),
    "SVM": SVR()
}

# Configurar el experimento en MLflow
mlflow.set_experiment("/Shared/Prediccion_Viviendas_California") # <-- Nota para estudiantes: MLflow creará esto por defecto si se omite

print("Iniciando entrenamiento y registro en MLflow...")

mejor_r2 = -float("inf")
mejor_nombre = ""
mejor_modelo = None

for nombre, modelo in modelos.items():
    with mlflow.start_run(run_name=nombre):
        # 1. Entrenar el modelo
        modelo.fit(X_train_scaled, y_train)
        
        # 2. Predecir
        predicciones = modelo.predict(X_test_scaled)
        
        # 3. Calcular métricas
        mse = mean_squared_error(y_test, predicciones)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, predicciones)
        
        # 4. Registrar en MLflow
        mlflow.log_param("modelo_tipo", nombre)
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        mlflow.sklearn.log_model(modelo, "modelo")
        
        print(f"Modelo: {nombre.ljust(20)} | RMSE: {rmse:.4f} | R2: {r2:.4f}")
        
        # Guardar en memoria el mejor modelo manual para el final
        if r2 > mejor_r2:
            mejor_r2 = r2
            mejor_nombre = nombre
            mejor_modelo = modelo

print(f"\n¡Entrenamiento finalizado! El mejor modelo manual fue: {mejor_nombre} con un R2 de {mejor_r2:.4f}")
print("Revisa la pestaña 'Experiments' (Experimentos) a la derecha para ver la interfaz gráfica de MLflow.")

Iniciando entrenamiento y registro en MLflow...


2026/03/02 00:54:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

Modelo: Linear_Regression    | RMSE: 0.7456 | R2: 0.5758


2026/03/02 00:54:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

Modelo: Gradient_Boosting    | RMSE: 0.5798 | R2: 0.7435


2026/03/02 00:55:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

Modelo: SVM                  | RMSE: 0.5975 | R2: 0.7276

¡Entrenamiento finalizado! El mejor modelo manual fue: Random_Forest con un R2 de 0.8039
Revisa la pestaña 'Experiments' (Experimentos) a la derecha para ver la interfaz gráfica de MLflow.


# Celda 3: Entrenamiento con AutoGluon (AutoML) e integración con MLflow


In [0]:
%pip install autogluon -q
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from autogluon.tabular import TabularPredictor
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
import mlflow

print("Preparando datos para AutoGluon...")
# AutoGluon prefiere los datos originales (sin escalar) en un DataFrame con la variable objetivo
train_data = X_train.copy()
train_data['Precio_Vivienda'] = y_train

test_data = X_test.copy()
test_data['Precio_Vivienda'] = y_test

# Iniciamos un nuevo "Run" en MLflow para que quede registrado con los demás
with mlflow.start_run(run_name="AutoGluon_AutoML"):
    print("Iniciando entrenamiento con AutoGluon (Límite: 2 minutos)...")
    
    # 1. Entrenar el modelo AutoML
    predictor = TabularPredictor(label='Precio_Vivienda', path='autogluon_modelos').fit(
        train_data, 
        time_limit=120, # 2 minutos de límite para la clase
        presets='medium_quality',
        verbosity=2
    )
    
    # 2. Hacer predicciones con el conjunto de prueba
    predicciones_ag = predictor.predict(test_data)
    
    # 3. Calcular las mismas métricas que usamos en los modelos manuales
    mse_ag = mean_squared_error(y_test, predicciones_ag)
    rmse_ag = np.sqrt(mse_ag)
    r2_ag = r2_score(y_test, predicciones_ag)
    
    # CORRECCIÓN: Usamos la propiedad model_best en lugar de get_model_best()
    mejor_modelo_interno = predictor.model_best
    
    # 4. Registrar en MLflow
    mlflow.log_param("modelo_tipo", "AutoGluon_Ensemble")
    mlflow.log_param("mejor_algoritmo_interno", mejor_modelo_interno)
    mlflow.log_metric("mse", mse_ag)
    mlflow.log_metric("rmse", rmse_ag)
    mlflow.log_metric("r2", r2_ag)
    
    print("\n¡AutoGluon ha terminado y se ha registrado en MLflow!")
    print(f"Mejor modelo interno de AutoGluon: {mejor_modelo_interno}")
    print(f"RMSE: {rmse_ag:.4f} | R2: {r2_ag:.4f}")
    
    print("\nRanking de todos los modelos probados por AutoGluon:")
    # Mostrar la tabla comparativa de AutoGluon
    leaderboard = predictor.leaderboard(test_data)
    display(leaderboard)

print("\n¡Ve a la pestaña de MLflow! Podrás comparar el R2 de AutoGluon directamente contra tu mejor modelo manual.")

Preparando datos para AutoGluon...


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   aarch64
Platform Version:   #107-Ubuntu SMP Thu Jan 22 13:39:51 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       5.40 GB / 15.28 GB (35.3%)
Disk Space Avail:   10.00 GB / 10.00 GB (100.0%)
	We recommend a minimum available disk space of 10 GB, and large datasets may require more.
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'


Iniciando entrenamiento con AutoGluon (Límite: 2 minutos)...


Beginning AutoGluon training ... Time limit = 120s
AutoGluon will save models to "/Workspace/Users/olbustosa@gmail.com/Drafts/autogluon_modelos"
Train Data Rows:    16512
Train Data Columns: 8
Label Column:       Precio_Vivienda
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and many unique label-values observed).
	Label info (max, min, mean, stddev): (5.00001, 0.14999, 2.07195, 1.15623)
	If 'regression' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    5532.26 MB
	Train Data (Original)  Memory Usage: 1.01 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature

[1000]	valid_set's rmse: 0.490271
[2000]	valid_set's rmse: 0.480824
[3000]	valid_set's rmse: 0.479103
[4000]	valid_set's rmse: 0.47806
[5000]	valid_set's rmse: 0.477555
[6000]	valid_set's rmse: 0.478424


...

[1000]	valid_set's rmse: 0.45723
[2000]	valid_set's rmse: 0.455822
[3000]	valid_set's rmse: 0.454135


	-0.454	 = Validation score   (-root_mean_squared_error)
	3.34s	 = Training   runtime
	0.16s	 = Validation runtime
Fitting model: RandomForestMSE ... Training model for up to 109.40s of the 109.40s of remaining time.
	Fitting with cpus=2, gpus=0, mem=0.1/5.4 GB
	-0.5303	 = Validation score   (-root_mean_squared_error)
	20.38s	 = Training   runtime
	0.17s	 = Validation runtime
Fitting model: CatBoost ... Training model for up to 87.73s of the 87.73s of remaining time.
	Fitting with cpus=2, gpus=0
	-0.4356	 = Validation score   (-root_mean_squared_error)
	67.64s	 = Training   runtime
	0.01s	 = Validation runtime
Fitting model: ExtraTreesMSE ... Training model for up to 19.83s of the 19.82s of remaining time.
	Fitting with cpus=2, gpus=0, mem=0.1/5.4 GB
	-0.5307	 = Validation score   (-root_mean_squared_error)
	4.8s	 = Training   runtime
	0.17s	 = Validation runtime
Fitting model: NeuralNetFastAI ... Training model for up to 13.60s of the 13.60s of remaining time.
	Fitting with cpus=2, gp


¡AutoGluon ha terminado y se ha registrado en MLflow!
Mejor modelo interno de AutoGluon: WeightedEnsemble_L2
RMSE: 0.4296 | R2: 0.8591

Ranking de todos los modelos probados por AutoGluon:


model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
WeightedEnsemble_L2,-0.4296240131749733,-0.4346898645265652,root_mean_squared_error,0.7918004989624023,0.19056415557861328,84.36196804046631,0.002858400344848633,7.410049438476562E-4,0.009561538696289062,2,true,7
CatBoost,-0.43115762862183904,-0.435550314918031,root_mean_squared_error,0.10286211967468262,0.014870166778564453,67.63907241821289,0.10286211967468262,0.014870166778564453,67.63907241821289,1,true,4
LightGBM,-0.43810927497471297,-0.4540054342004592,root_mean_squared_error,0.5040585994720459,0.15792226791381836,3.3355298042297363,0.5040585994720459,0.15792226791381836,3.3355298042297363,1,true,2
LightGBMXT,-0.4631012875779992,-0.4774551217990905,root_mean_squared_error,0.9039878845214844,0.34336233139038086,5.964994430541992,0.9039878845214844,0.34336233139038086,5.964994430541992,1,true,1
RandomForestMSE,-0.5065815593450295,-0.5302851737954969,root_mean_squared_error,1.200695276260376,0.16597390174865723,20.379114389419556,1.200695276260376,0.16597390174865723,20.379114389419556,1,true,3
ExtraTreesMSE,-0.5144809146954546,-0.5306570609938839,root_mean_squared_error,0.835068941116333,0.1681499481201172,4.796033620834351,0.835068941116333,0.1681499481201172,4.796033620834351,1,true,5
NeuralNetFastAI,-0.6243627127120611,-0.6110287717806588,root_mean_squared_error,0.1820213794708252,0.017030715942382812,13.377804279327393,0.1820213794708252,0.017030715942382812,13.377804279327393,1,true,6



¡Ve a la pestaña de MLflow! Podrás comparar el R2 de AutoGluon directamente contra tu mejor modelo manual.


# Celda 4: Empaquetar el modelo de Scikit-Learn como Pickle (Ajustada)
Mantenemos esta celda enfocada en el modelo de scikit-learn de la Celda 2.

In [0]:
import joblib

# NOTA PARA LA CLASE: 
# Aunque AutoGluon logró resultados increíbles, exporta una estructura de carpetas compleja.
# Para nuestro primer despliegue en un servidor web (API con Flask), usaremos 
# nuestro mejor modelo manual de Scikit-Learn por su simplicidad (un solo archivo .pkl).

artefactos = {
    "modelo": mejor_modelo,
    "escalador": scaler
}

archivo_salida = "mejor_modelo_california.pkl"
joblib.dump(artefactos, archivo_salida)

print(f"¡Modelo de Scikit-Learn empaquetado y guardado como '{archivo_salida}'!")

¡Modelo de Scikit-Learn empaquetado y guardado como 'mejor_modelo_california.pkl'!
